# Figure 3 | Utah FORGE gravity inversion

Run the cells in order from the repository root or `examples/`. Panels are saved to `examples/Figure/fig3/` at 400 dpi. The source field data and saved case results must be obtained separately. The assembled manuscript figure is not generated here.

Panel a is the manually assembled workflow schematic; this notebook exports the data panels b–e.

## Setup

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.tri as mtri
from matplotlib.colors import LightSource, LogNorm, Normalize, TwoSlopeNorm
from matplotlib.cm import ScalarMappable
from matplotlib.ticker import MaxNLocator
from matplotlib.patches import Patch
from mpl_toolkits.axes_grid1 import make_axes_locatable
from mpl_toolkits.mplot3d.art3d import Line3DCollection
from scipy.interpolate import griddata
from scipy.spatial import cKDTree
from PIL import Image

def find_project_root():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / 'examples').is_dir() and (candidate / 'pyproject.toml').is_file():
            return candidate
    raise FileNotFoundError('Run this notebook from the project root or examples directory.')


ROOT = find_project_root()
OUT = ROOT / 'examples' / 'Figure' / 'fig3'
OUT.mkdir(parents=True, exist_ok=True)

STATE = json.loads((ROOT / 'Result' / 'Utah-FORGE' / 'case_state.json').read_text(encoding='utf-8'))
RUNS = ROOT / 'Result' / 'Utah-FORGE' / 'runs'
GRAVITY_RUN = RUNS / STATE['run_id']
TEM_RUN = RUNS / STATE['tem_run_id']
MT_RUN = RUNS / STATE['mt_run_id']
IMPORTS = ROOT / 'Result' / 'Utah-FORGE' / 'imports'
GEO_MODEL = IMPORTS / 'geological_model.npz'
MODEL_EXTENT_M = (331_900.0, 340_150.0, 4_259_925.0, 4_265_150.0)
ELEVATION_RANGE_M = (-1_200.0, 2_490.0)

mpl.rcParams.update({
    'font.family': 'Arial', 'font.size': 11, 'axes.linewidth': 1.1,
    'axes.labelsize': 14, 'xtick.labelsize': 11, 'ytick.labelsize': 11,
    'axes.grid': False, 'legend.frameon': False,
    'savefig.facecolor': 'white', 'figure.facecolor': 'white',
})
RES_CMAP = mpl.colormaps['turbo'].copy(); RES_CMAP.set_bad('white')
RES_NORM = LogNorm(1.0, 1.0e4)

def bold_ticks(ax):
    ax.xaxis.label.set_fontweight('bold'); ax.yaxis.label.set_fontweight('bold')
    for tick in ax.get_xticklabels() + ax.get_yticklabels():
        tick.set_fontweight('bold')
    ax.tick_params(width=1.2, length=5)

def reduce_xy_ticks(ax, nbins=4):
    ax.xaxis.set_major_locator(MaxNLocator(nbins=nbins, min_n_ticks=3))
    ax.yaxis.set_major_locator(MaxNLocator(nbins=nbins, min_n_ticks=3))

def add_resistivity_colorbar(fig, ax, mappable, pad=0.14):
    divider = make_axes_locatable(ax)
    cax = divider.append_axes('right', size='4.5%', pad=pad)
    cb = fig.colorbar(mappable, cax=cax)
    cb.set_ticks([1, 10, 100, 1000, 10000])
    cb.set_ticklabels([r'$10^0$', r'$10^1$', r'$10^2$', r'$10^3$', r'$10^4$'])
    cb.set_label(r'Resistivity ($\Omega$ m)', fontsize=14, fontweight='bold')
    cb.ax.tick_params(labelsize=11, width=1.1)
    for tick in cb.ax.get_yticklabels(): tick.set_fontweight('bold')
    return cb

# Fixed physical plot-frame geometry for panels b--d.  The colorbar in d is extra.
SECTION_FRAME_W_IN = 6.40
SECTION_FRAME_H_IN = 2.16
SECTION_LEFT_IN = 1.12
SECTION_BOTTOM_IN = 0.86
SECTION_TOP_IN = 0.24
SECTION_RIGHT_IN = 0.22
SECTION_CBAR_GAP_IN = 0.20
SECTION_CBAR_W_IN = 0.29
SECTION_CBAR_RIGHT_IN = 1.18

def section_canvas(with_colorbar=False):
    extra = (SECTION_CBAR_GAP_IN + SECTION_CBAR_W_IN + SECTION_CBAR_RIGHT_IN) if with_colorbar else SECTION_RIGHT_IN
    width = SECTION_LEFT_IN + SECTION_FRAME_W_IN + extra
    height = SECTION_BOTTOM_IN + SECTION_FRAME_H_IN + SECTION_TOP_IN
    fig = plt.figure(figsize=(width, height), facecolor='white')
    ax = fig.add_axes([
        SECTION_LEFT_IN / width, SECTION_BOTTOM_IN / height,
        SECTION_FRAME_W_IN / width, SECTION_FRAME_H_IN / height,
    ])
    cax = None
    if with_colorbar:
        cbar_left = SECTION_LEFT_IN + SECTION_FRAME_W_IN + SECTION_CBAR_GAP_IN
        cax = fig.add_axes([
            cbar_left / width, SECTION_BOTTOM_IN / height,
            SECTION_CBAR_W_IN / width, SECTION_FRAME_H_IN / height,
        ])
    return fig, ax, cax

def finish_resistivity_colorbar(fig, cax, mappable):
    cb = fig.colorbar(mappable, cax=cax)
    cb.set_ticks([1, 10, 100, 1000, 10000])
    cb.set_ticklabels([r'$10^0$', r'$10^1$', r'$10^2$', r'$10^3$', r'$10^4$'])
    cb.set_label(r'Resistivity ($\Omega$ m)', fontsize=14, fontweight='bold')
    cb.ax.tick_params(labelsize=11, width=1.2, length=5)
    for tick in cb.ax.get_yticklabels():
        tick.set_fontweight('bold')
    return cb

def save_fixed_panel(fig, filename, dpi=400):
    path = OUT / filename
    fig.savefig(path, dpi=dpi, facecolor='white')
    # Optional vector copy for rendered-text QA; off unless the variable is set.
    qa_dir = __import__('os').environ.get('FIG3_QA_PDF_DIR')
    if qa_dir:
        fig.savefig(Path(qa_dir) / Path(filename).with_suffix('.pdf'), facecolor='white')
    plt.close(fig)
    return path

def save_png(fig, filename, dpi=400):
    path = OUT / filename
    fig.savefig(path, dpi=dpi, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    return path

## b | Observed gravity and residual

In [ ]:
from matplotlib.colors import TwoSlopeNorm
from matplotlib.ticker import FuncFormatter

gravity_obs = pd.read_csv(ROOT / STATE['observations_path'])
gravity_pred = np.load(GRAVITY_RUN / 'predicted_gravity_mgal.npy')
gravity_residual = gravity_obs.gravity_mgal.to_numpy() - gravity_pred      # observed minus predicted
gravity_summary = json.loads((GRAVITY_RUN / 'result.json').read_text(encoding='utf-8'))['summary']
rms = float(np.sqrt(np.mean(gravity_residual ** 2)))
print(f'{len(gravity_obs)} stations; chi-squared {gravity_summary["chi_squared"]:.2f}; '
      f'residual RMS {rms:.3f} mGal, max |{np.abs(gravity_residual).max():.3f}| mGal')

# A light hillshade of the 50 m terrain, for orientation only.
terrain = pd.read_csv(IMPORTS / 'terrain_50m.csv')
tx, ty = np.unique(terrain.x_m), np.unique(terrain.y_m)
tz = terrain.pivot_table(index='y_m', columns='x_m', values='z_m').reindex(index=ty, columns=tx).to_numpy()
tz = np.where(np.isfinite(tz), tz, np.nanmedian(tz))
shade = LightSource(azdeg=315, altdeg=40).hillshade(tz, vert_exag=2.0, dx=50.0, dy=50.0)
terrain_extent = [tx[0] / 1000, tx[-1] / 1000, ty[0] / 1000, ty[-1] / 1000]

MAP_FRAME_W_IN = SECTION_FRAME_W_IN
MAP_FRAME_H_IN = MAP_FRAME_W_IN * (MODEL_EXTENT_M[3] - MODEL_EXTENT_M[2]) / (MODEL_EXTENT_M[1] - MODEL_EXTENT_M[0])
# Panel b is read at a larger type size than b--d, so it keeps its own margins.
G_TICK_PT, G_LABEL_PT, G_CBAR_TICK_PT, G_CBAR_LABEL_PT, G_TEXT_PT = 21, 24, 21, 23, 21
G_LEFT_IN, G_BOTTOM_IN, G_TOP_IN, MAP_GAP_IN = 2.05, 1.40, 0.30, 0.45
G_CBAR_GAP_IN, G_CBAR_W_IN, G_CBAR_RIGHT_IN = 0.22, 0.32, 1.95
width = G_LEFT_IN + MAP_FRAME_W_IN + G_CBAR_GAP_IN + G_CBAR_W_IN + G_CBAR_RIGHT_IN
height = G_BOTTOM_IN + 2 * MAP_FRAME_H_IN + MAP_GAP_IN + G_TOP_IN
fig = plt.figure(figsize=(width, height), facecolor='white')
km = lambda v: np.asarray(v, float) / 1000.0

rows = (
    ('top', gravity_obs.gravity_mgal.to_numpy(), TwoSlopeNorm(vcenter=0.0, vmin=-12.5, vmax=12.5),
     [-12, -6, 0, 6, 12], 'mGal'),
    ('bottom', gravity_residual, TwoSlopeNorm(vcenter=0.0, vmin=-0.2, vmax=0.2),
     [-0.2, -0.1, 0.0, 0.1, 0.2], 'mGal'),
)
# Gridded for display (imagesc-style): linear between stations on a 50 m grid, and the
# nearest station's value outside their hull, so the frame is filled.
grid_x = np.arange(MODEL_EXTENT_M[0], MODEL_EXTENT_M[1] + 1.0, 50.0)
grid_y = np.arange(MODEL_EXTENT_M[2], MODEL_EXTENT_M[3] + 1.0, 50.0)
GXg, GYg = np.meshgrid(grid_x, grid_y)
station_xy = gravity_obs[['x_m', 'y_m']].to_numpy()
axes = []
for position, values, norm, ticks, label in rows:
    y0 = G_BOTTOM_IN + (MAP_FRAME_H_IN + MAP_GAP_IN if position == 'top' else 0.0)
    ax = fig.add_axes([G_LEFT_IN / width, y0 / height, MAP_FRAME_W_IN / width, MAP_FRAME_H_IN / height])
    cax = fig.add_axes([(G_LEFT_IN + MAP_FRAME_W_IN + G_CBAR_GAP_IN) / width, y0 / height,
                        G_CBAR_W_IN / width, MAP_FRAME_H_IN / height])
    ax.imshow(shade, extent=terrain_extent, origin='lower', cmap='gray', vmin=0.0, vmax=1.0, alpha=0.35, zorder=0)
    gridded = griddata(station_xy, values, (GXg, GYg), method='linear')
    gridded = np.where(np.isfinite(gridded), gridded, griddata(station_xy, values, (GXg, GYg), method='nearest'))
    points = ax.imshow(gridded, extent=[km(grid_x[0]) - 0.025, km(grid_x[-1]) + 0.025,
                                        km(grid_y[0]) - 0.025, km(grid_y[-1]) + 0.025],
                       origin='lower', cmap='RdBu_r', norm=norm, interpolation='nearest', zorder=2)
    ax.set_xlim(km(MODEL_EXTENT_M[0]), km(MODEL_EXTENT_M[1]))
    ax.set_ylim(km(MODEL_EXTENT_M[2]), km(MODEL_EXTENT_M[3]))
    ax.set_aspect('equal', adjustable='box')
    ax.set_ylabel('Northing (km)', fontsize=G_LABEL_PT)
    reduce_xy_ticks(ax)
    if position == 'top':
        plt.setp(ax.get_xticklabels(), visible=False)
    else:
        ax.set_xlabel('Easting (km)', fontsize=G_LABEL_PT)
        ax.text(0.02, 0.04, f'RMS = {rms:.3f} mGal',
                transform=ax.transAxes, ha='left', va='bottom', fontsize=G_TEXT_PT, fontweight='bold',
                bbox=dict(facecolor='white', edgecolor='none', alpha=0.85, pad=2.0), zorder=5)
    cb = fig.colorbar(points, cax=cax)
    cb.set_ticks(ticks)
    cb.ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f'{v:g}'))
    cb.set_label(label, fontsize=G_CBAR_LABEL_PT, fontweight='bold')
    cb.ax.tick_params(labelsize=G_CBAR_TICK_PT, width=1.4, length=7)
    for tick in cb.ax.get_yticklabels():
        tick.set_fontweight('bold')
    bold_ticks(ax)
    ax.tick_params(labelsize=G_TICK_PT, width=1.4, length=7)
    axes.append(ax)
save_fixed_panel(fig, 'Fig3b_gravity_data_residual.png')

## c | Seismic basement prior

In [ ]:
granite = np.loadtxt(ROOT / STATE['granite_path'], delimiter=',', skiprows=1)
pad = 300.0
near = ((granite[:, 0] >= MODEL_EXTENT_M[0] - pad) & (granite[:, 0] <= MODEL_EXTENT_M[1] + pad)
        & (granite[:, 1] >= MODEL_EXTENT_M[2] - pad) & (granite[:, 1] <= MODEL_EXTENT_M[3] + pad))
gx = np.arange(MODEL_EXTENT_M[0], MODEL_EXTENT_M[1] + 1.0, 50.0)
gy = np.arange(MODEL_EXTENT_M[2], MODEL_EXTENT_M[3] + 1.0, 50.0)
GX, GY = np.meshgrid(gx, gy)
top_of_granite = griddata(granite[near, :2], granite[near, 2], (GX, GY), method='linear')
print(f'top of granite in the model extent: {np.nanmin(top_of_granite):.0f} to {np.nanmax(top_of_granite):.0f} m elevation')

fig, ax, cax = section_canvas(with_colorbar=True)
fig.set_size_inches(fig.get_size_inches()[0], SECTION_BOTTOM_IN + MAP_FRAME_H_IN + SECTION_TOP_IN)
w_in, h_in = fig.get_size_inches()
ax.set_position([SECTION_LEFT_IN / w_in, SECTION_BOTTOM_IN / h_in, MAP_FRAME_W_IN / w_in, MAP_FRAME_H_IN / h_in])
cax.set_position([(SECTION_LEFT_IN + MAP_FRAME_W_IN + SECTION_CBAR_GAP_IN) / w_in, SECTION_BOTTOM_IN / h_in,
                  SECTION_CBAR_W_IN / w_in, MAP_FRAME_H_IN / h_in])
surface_norm = Normalize(-1500.0, 2000.0)
image = ax.pcolormesh(km(gx), km(gy), top_of_granite, cmap='cividis', norm=surface_norm,
                      shading='nearest', rasterized=True)
levels = np.arange(-1500.0, 2001.0, 250.0)
lines = ax.contour(km(gx), km(gy), top_of_granite, levels=levels, colors='#1A1A1A', linewidths=0.55, alpha=0.75)
ax.clabel(lines, levels=[-1000.0, 0.0, 1000.0], fmt='%d', fontsize=8.5, inline=True)
ax.set_xlim(km(MODEL_EXTENT_M[0]), km(MODEL_EXTENT_M[1]))
ax.set_ylim(km(MODEL_EXTENT_M[2]), km(MODEL_EXTENT_M[3]))
ax.set_aspect('equal', adjustable='box')
ax.set_xlabel('Easting (km)'); ax.set_ylabel('Northing (km)')
reduce_xy_ticks(ax)
cb = fig.colorbar(image, cax=cax)
cb.set_ticks([-1500, -500, 500, 1500])
cb.set_label('Top of granite elevation (m)', fontsize=13, fontweight='bold')
cb.ax.tick_params(labelsize=11, width=1.2, length=5)
for tick in cb.ax.get_yticklabels():
    tick.set_fontweight('bold')
bold_ticks(ax)
save_fixed_panel(fig, 'Fig3c_seismic_basement.png')

## d–e | Recovered density volume and sections

In [ ]:
# Panel d: density cutaway from the saved inversion mesh.
obs = pd.read_csv(ROOT / STATE['observations_path'])
topo = pd.read_csv(IMPORTS / 'terrain_50m.csv')
mesh = np.load(GRAVITY_RUN / 'mesh_geometry.npz')
centres = mesh['cell_centers_m'][mesh['active_cells'].astype(bool)]
density_model = np.load(GRAVITY_RUN / 'recovered_density_g_cc.npy')
density_tree = cKDTree(centres)

x0, y0 = np.median(obs.x_m), np.median(obs.y_m)
x = np.linspace(topo.x_m.min(), topo.x_m.max(), 100)
y = np.linspace(topo.y_m.min(), topo.y_m.max(), 100)
xx, yy = np.meshgrid(x, y)
top = griddata((topo.x_m, topo.y_m), topo.z_m, (xx, yy), method='linear')
top = np.where(np.isfinite(top), top, np.nanmedian(topo.z_m))
depth = 1500.0
depths = np.linspace(0.0, 3300.0, 72)

def nearest_density(a, b, c):
    points = np.c_[a.ravel(), b.ravel(), c.ravel()]
    return density_model[density_tree.query(points)[1]].reshape(a.shape)

def terrain_profile(a, b):
    values = griddata((topo.x_m, topo.y_m), topo.z_m, (a, b), method='linear')
    return np.where(np.isfinite(values), values, np.nanmedian(topo.z_m))

def density_surface(a, b, levels=None):
    levels = depths if levels is None else levels
    z = terrain_profile(a[0], b[0])[None, :] - levels[:, None]
    return z, nearest_density(a, b, z)

def surface_x(y_value):
    a, _ = np.meshgrid(x, depths)
    b = np.full_like(a, y_value)
    z, values = density_surface(a, b)
    return a, b, z, values

def surface_y(x_value):
    b, _ = np.meshgrid(y, depths)
    a = np.full_like(b, x_value)
    z, values = density_surface(a, b)
    return a, b, z, values

# The cutaway samples more finely than the sections, on a grid of its own.
xa = np.linspace(topo.x_m.min(), topo.x_m.max(), 240)
ya = np.linspace(topo.y_m.min(), topo.y_m.max(), 180)
xxa, yya = np.meshgrid(xa, ya)
topa = terrain_profile(xxa, yya)
depths_a = np.linspace(0.0, 3300.0, 160)
A_TICK_PT, A_LABEL_PT, A_CBAR_TICK_PT, A_CBAR_LABEL_PT = 11, 14, 20, 23
fig = plt.figure(figsize=(9.5, 7.0), facecolor='white')
ax = fig.add_axes([0.0, 0.0, 0.74, 1.0], projection='3d')
ax.set_proj_type('persp', focal_length=0.45)
ax.view_init(elev=26, azim=-58)
ax.set_box_aspect((1.6, 1.0, 0.62), zoom=1.05)
density_norm = TwoSlopeNorm(vmin=-0.22, vcenter=0.0, vmax=0.22)
density_cmap = mpl.colormaps['coolwarm']
terrain_norm = Normalize(1200.0, 3000.0)
cut = (xxa >= x0) & (yya <= y0)
surface_z = topa.copy()
surface_z[cut] = np.nan
ax.plot_surface(xxa / 1000, yya / 1000, surface_z / 1000,
                facecolors=mpl.colormaps['terrain'](terrain_norm(topa)),
                linewidth=0, antialiased=False, shade=False, rcount=len(ya), ccount=len(xa))
floor_z = (topa - depth).copy()
# One sample wider than the cut, so the floor meets the faces without a seam.
floor = (xxa >= x0 - np.diff(xa)[0]) & (yya <= y0 + np.diff(ya)[0])
floor_z[~floor] = np.nan
ax.plot_surface(xxa / 1000, yya / 1000, floor_z / 1000,
                facecolors=density_cmap(density_norm(nearest_density(xxa, yya, topa - depth))),
                linewidth=0, antialiased=False, shade=False, rcount=len(ya), ccount=len(xa))
# Faces, as (fixed axis, coordinate, positions along it, depth range).
upper = depths_a[depths_a <= depth]
lower = depths_a[depths_a >= depth]
# Each piece carries the cut coordinate itself, so neighbouring pieces meet rather than
# leaving a seam of background between them.
xw, xe = np.r_[xa[xa < x0], x0], np.r_[x0, xa[xa > x0]]
ys_, yn = np.r_[y0, ya[ya > y0]], np.r_[ya[ya < y0], y0]
faces = [
    ('y', ya.min(), xw, depths_a), ('y', ya.min(), xe, lower),
    ('x', xa.max(), ys_, depths_a), ('x', xa.max(), yn, lower),
    ('y', y0, xe, upper), ('x', x0, yn, upper),
]
for fixed, coordinate, along, levels in faces:
    a, _ = np.meshgrid(along, levels)
    if fixed == 'y':
        b = np.full_like(a, coordinate)
    else:
        b, a = a, np.full_like(a, coordinate)
    z, values = density_surface(a, b, levels)
    ax.plot_surface(a / 1000, b / 1000, z / 1000,
                    facecolors=density_cmap(density_norm(values)),
                    linewidth=0, antialiased=False, shade=False,
                    rcount=len(levels), ccount=len(along))
# The ground line of every face, white, so the cut reads as a cut.
def edge(xs, ys, below=0.0, width=1.6):
    ax.plot(xs / 1000, ys / 1000, (terrain_profile(xs, ys) - below) / 1000 + 0.01,
            color='white', linewidth=width, zorder=10)
edge(xw, np.full_like(xw, ya.min()))
edge(np.full_like(ys_, xa.max()), ys_)
edge(xe, np.full_like(xe, y0))
edge(np.full_like(yn, x0), yn)
ax.set_xlim(xa.min() / 1000, xa.max() / 1000)
ax.set_ylim(ya.min() / 1000, ya.max() / 1000)
ax.set_zlim(-1.8, 2.1)
ax.set_xticks([333, 336, 339])
ax.set_yticks([4260.0, 4262.5, 4265.0])
ax.set_zticks([-1, 0, 1, 2])
ax.yaxis.set_major_formatter(mpl.ticker.FormatStrFormatter('%.1f'))
for axis in (ax.xaxis, ax.yaxis, ax.zaxis):
    axis.set_pane_color((1, 1, 1, 0))
    axis.pane.set_edgecolor('#1A1A1A')
    axis._axinfo['grid'].update(color='#DDDDDD', linewidth=0.6)
    axis.line.set_linewidth(0.8)
ax.tick_params(labelsize=A_TICK_PT, pad=2)
for label in ax.get_xticklabels() + ax.get_yticklabels() + ax.get_zticklabels():
    label.set_fontweight('bold')
ax.set_xlabel('Easting (km)', fontsize=A_LABEL_PT, fontweight='bold', labelpad=10)
ax.set_ylabel('Northing (km)', fontsize=A_LABEL_PT, fontweight='bold', labelpad=12)
ax.set_zlabel('Elevation (km)', fontsize=A_LABEL_PT, fontweight='bold', labelpad=6)
cax = fig.add_axes([0.875, 0.14, 0.032, 0.68])
cb = fig.colorbar(ScalarMappable(norm=density_norm, cmap=density_cmap), cax=cax)
cb.set_ticks([-0.2, -0.1, 0.0, 0.1, 0.2])
cb.ax.tick_params(labelsize=A_CBAR_TICK_PT)
cb.set_label(r'Density contrast (g cm$^{-3}$)', fontsize=A_CBAR_LABEL_PT)
save_png(fig, 'Fig3d_density_cutaway_3D.png')

# Panel e sections: identical physical plot-frame dimensions and typography.
density_section_norm = TwoSlopeNorm(vmin=-0.22, vcenter=0.0, vmax=0.22)

def draw_density_section(horizontal_m, elevation_m, density_values, xlabel, filename):
    fig, ax, _ = section_canvas(with_colorbar=False)
    image = ax.pcolormesh(
        horizontal_m / 1000.0, elevation_m / 1000.0, density_values,
        shading='auto', cmap='coolwarm', norm=density_section_norm, rasterized=True,
    )
    ground = elevation_m[0] / 1000.0
    horizontal_km = horizontal_m[0] / 1000.0
    ax.plot(horizontal_km, ground, color='#161616', linewidth=1.2, zorder=4)
    ax.set_xlim(float(np.nanmin(horizontal_km)), float(np.nanmax(horizontal_km)))
    ax.set_ylim(float(np.nanmin(elevation_m) / 1000.0), float(np.nanmax(ground) + 0.10))
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Elevation (km)')
    reduce_xy_ticks(ax, nbins=4)
    bold_ticks(ax)
    save_fixed_panel(fig, filename)
    return image

ew = surface_x(y0)
ns = surface_y(x0)
draw_density_section(ew[0], ew[2], ew[3], 'Easting (km)', 'Fig3e_density_section_EW.png')
draw_density_section(ns[1], ns[2], ns[3], 'Northing (km)', 'Fig3e_density_section_NS.png')
for filename in ['Fig3e_density_section_EW.png', 'Fig3e_density_section_NS.png']:
    with Image.open(OUT / filename) as image:
        print(filename, image.size)


## Export check

Every listed panel must exist and have nonzero dimensions.

In [ ]:
from PIL import Image
expected = ['Fig3b_gravity_data_residual.png', 'Fig3c_seismic_basement.png', 'Fig3d_density_cutaway_3D.png', 'Fig3e_density_section_EW.png', 'Fig3e_density_section_NS.png']
for name in expected:
    path = OUT / name
    if not path.is_file():
        raise FileNotFoundError(path)
    with Image.open(path) as panel:
        assert panel.width > 0 and panel.height > 0, path
print(f'Figure panels checked: {len(expected)} in {OUT}')
